In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import datetime
import time
from sklearn.linear_model import LogisticRegression
import pyblp
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from sklearn.model_selection import KFold
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from numpy.random import *
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
import seaborn as sns
import scipy.stats
import warnings
from sklearn.exceptions import ConvergenceWarning
import sys
sys.path.append("src")

from data_generation import *
from neural_networks import *
from estimation import *
from train_varying_products import *

# Ignore ConvergenceWarning from sklearn
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.precision", 4)
pd.options.mode.chained_assignment = None  


In [2]:
def moment_func_varying(x_1, x_2, mask, model, delta):
    # apply price change 
    new_x_1 = x_1.clone()
    new_x_1[:,:,-1] = new_x_1[:,:,-1] + delta * new_x_1[:,:,-1] ## in data generation, the last feature is price
    
    # calculate moment
    a1 = model(x_2, new_x_1, mask)
    a0 = model(x_2, x_1, mask)
    moment = (a1 - a0) 
    
    return moment, a0
    

In [3]:
def alpha_loss_varying(x_1, x_2, mask, model, delta): 
    moment, alpha = moment_func_varying(x_1, x_2, mask, model, delta)
    return (alpha**2 - moment * 2).mean()


In [4]:
def train_alpha_varying(data, delta):
    K = data['K']
    x_1, x_2, mask = x_transform_mm_varying(data)
    model = SmallDeepSetVarying(x_d = K+1)
    model = model.cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    losses = []
    x_1, x_2, mask = torch.from_numpy(x_1).float().cuda(), torch.from_numpy(x_2).float().cuda(), torch.from_numpy(mask).float().cuda()

    for _ in range(5000):
        loss = alpha_loss_varying(x_1, x_2, mask, model, delta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if loss < 1e-8:
            break
        #print(loss)
    return model, losses

In [5]:
def pred_theta_varying(f_model, alpha_model, data, delta):
    x_1, x_2, mask = x_transform_mm_varying(data) 
    y = np.log(data['Y'])
    x_1, x_2, mask, y = torch.from_numpy(x_1).float().cuda(), torch.from_numpy(x_2).float().cuda(), torch.from_numpy(mask).float().cuda(), torch.from_numpy(y).float().cuda()
    
    ## the effect of price change on market share estimated using f_model 
    new_x_1 = x_1.clone()
    new_x_1[:,:,-1] = new_x_1[:,:,-1] + delta * new_x_1[:,:,-1]  ## in data generation, the last feature is price
    # calculate change in market share
    y1 = f_model(x_2, new_x_1, mask)
    y0 = f_model(x_2, x_1, mask)
    m_est = y1-y0
    ## alpha hat 
    alpha_hat = alpha_model(x_2, x_1, mask) 
    theta = m_est + alpha_hat * (y-y0)
    
    return m_est, theta, alpha_hat, y0
    

In [6]:
def Inference_varying(data, train_market,  delta):
    
    data_f1, data_f2 = split_train_test_blp(data, train_market)
    
    ### step 2: estimation by each fold of data 
    ## estimation of fold 2
    ## train on the left out data
    f_model, loss_f = train_deep_varying(data_f1)
    alpha_model, loss_alpha = train_alpha_varying(data_f1, delta)
    ## apply the trained f and alpha to the fold
    m_est1, theta1, alpha_hat1, f_hat1 = pred_theta_varying(f_model, alpha_model, data_f2, delta)
    
    ## estimation of fold 1
    ## train on the left out data
    f_model, loss_f = train_deep_varying(data_f2)
    alpha_model, loss_alpha = train_alpha_varying(data_f2, delta)
    ## apply the trained f and alpha to the fold
    m_est2, theta2, alpha_hat2, f_hat2 = pred_theta_varying(f_model, alpha_model, data_f1, delta)
    
    ### step 3: report the estimation result
    theta = torch.cat((theta1, theta2), dim=0)
    theta_hat = theta.mean()
    theta_sd = theta.std()
    
    return theta, theta_hat.cpu().detach().numpy(), theta_sd.cpu().detach().numpy()

In [7]:
def PlugIn_theta_blp(data,  delta):
    
    m1_deep, losses_deep = train_deep_varying(data)
    theta_deep = pred_theta_nc_blp(data, m1_deep, delta)
    theta_hat, theta_sd = Inference_varying(data,  delta)
    
    return [[theta_hat, theta_sd], theta_deep] #,theta_rcl, theta_mnl, theta_single


def pred_theta_nc_blp(data, model, delta =1):
    x_1, x_2, mask = x_transform_mm_varying(data) 
    x_1, x_2, mask = torch.from_numpy(x_1).float().cuda(), torch.from_numpy(x_2).float().cuda(), torch.from_numpy(mask).float().cuda()
    new_x_1 = x_1.clone()
    new_x_1[:,:,-1] = new_x_1[:,:,-1] + delta * new_x_1[:,:,-1]
    y_pred_new = model(x_2, new_x_1, mask)
    y_pred_old = model(x_2, x_1, mask)
    
    return (y_pred_new - y_pred_old).cpu().detach().numpy()

In [8]:
def cal_cover(true, theta_hat, theta_sd, J, M):
    ci_l = theta_hat - 1.96 * theta_sd / np.sqrt(J * M)
    ci_u = theta_hat + 1.96 * theta_sd / np.sqrt(J * M)
    cover = (ci_l < true) & (ci_u >  true)
    significant = ci_u < 0
    bias = np.abs(theta_hat - true)
    
    return np.mean(bias), np.mean(cover), np.mean(significant)

In [9]:
def split_train_test_blp(data, train_market):
    M = data['M']
    J = data['J']
    
    train_size = data['J_list'][train_market]
    
    data_train = {
        'X': data['X'].iloc[0: train_size],
        'Y': data['Y'][0: train_size],
        'M': train_market,
        'J': J, 
        'K': data['K'], 
        'params': data['params'], 
        #'generation_seed': data['generation_seed'], 
        'J_list': data['J_list'][0:(train_market+1)],
        'market_id' : data['market_id'][0: train_size]}
        
    data_test = {
        'X': data['X'].iloc[train_size: ,].reset_index(drop = True),
        'Y': data['Y'][train_size: ],
        'M': data['M']-train_market,
        'J': J, 
        'K': data['K'], 
        'params': data['params'], 
        #'generation_seed': data['generation_seed'], 
        'J_list': [x - train_size for x in blp_data['J_list'][train_market:]],
        'market_id' : data['market_id'][train_size:]}
    
    #print(data_test['M'])
    if 'b_random' in data.keys():
        data_train['b_random'] = [arr[0:train_size,:] for arr in data['b_random']]
        data_test['b_random'] = [arr[train_size:,:] for arr in data['b_random']]
    
    return data_train, data_test

In [10]:
data_file_name = 'blp_data_deep' +".pickle"
with open(data_file_name, 'rb') as f:
    blp_data = pickle.load(f) 

In [11]:
train_market = 11
delta = 0.1
theta_c, theta_c_mean, theta_c_sd = Inference_varying(blp_data, train_market,  delta)

In [12]:
theta_c = theta_c.cpu().detach().numpy()

In [13]:
theta_c.mean()

-0.35637453

In [14]:
df = blp_data['X'].copy()
df['market_id'] = blp_data['market_id']
df['theta'] = theta_c

In [17]:
df['elas'] = df['theta']/0.1

In [18]:
df['type'] = 'low'
df.loc[df.price > 8, 'type'] = 'medium'
df.loc[df.price > 20, 'type'] = 'high'
df['type'].value_counts()

medium    1053
low        903
high       261
Name: type, dtype: int64

,0,1,2,3,4,price,market_id,theta,elas,type
0,0.5290,0,1.8881,1.1502,-4.9415,4.9358,0,-1.1380,-11.3796,low
1,0.4943,0,1.9360,1.2780,-4.2525,5.5160,0,0.2249,2.2491,low
2,0.4676,0,1.7168,1.4592,-3.1718,7.1086,0,-0.0885,-0.8847,low
3,0.4265,0,1.6879,1.6068,-3.5115,6.8395,0,-0.2201,-2.2014,low
4,0.4525,0,1.5043,1.6458,-1.8475,8.9284,0,-0.2325,-2.3250,medium
...,...,...,...,...,...,...,...,...,...,...
2212,0.3859,1,2.6391,1.3056,2.9906,16.1400,19,-0.1843,-1.8431,medium
2213,0.4360,1,2.1364,1.3056,12.8375,25.9870,19,-0.2640,-2.6398,high
2214,0.3583,0,3.5188,0.8437,-9.9514,3.3933,19,-0.3812,-3.8123,low
2215,0.8149,1,3.0162,1.0939,31.5175,44.7590,19,0.0183,0.1825,high


In [42]:
high_samples

,,0,1,2,3,4,price,market_id,theta,elas,type
market_id,,,,,,,,,,,
0,43,0.7296,0,1.9705,1.7360,13.1700,21.7605,0,0.0682,0.6822,high
1,134,0.4424,1,1.3107,1.7680,11.1324,20.6699,1,-0.3902,-3.9020,high
2,252,0.4944,1,1.8664,1.2283,22.0673,31.3784,2,-0.1228,-1.2280,high
3,326,0.6177,0,1.7107,1.0816,15.9423,24.0872,3,-0.4359,-4.3593,high
4,416,0.3779,1,1.4233,1.2950,12.5839,23.7100,4,-0.6626,-6.6265,high
5,442,0.4253,1,1.4466,1.4688,13.6789,21.9315,5,-0.2197,-2.1973,high
6,608,0.4586,1,1.2252,1.5540,28.7984,41.6518,6,-0.2797,-2.7970,high
7,706,0.7430,0,1.4597,1.0985,20.2979,31.8635,7,-0.2189,-2.1891,high
8,800,0.2013,0,2.1708,1.3370,11.3152,23.0992,8,-0.4836,-4.8365,high


In [23]:
medium_theta_hat = medium_samples['elas'].mean() 
medium_theta_sd = medium_samples['elas'].std() 
medium_n = len(medium_samples['elas'])
[medium_theta_hat - 1.96 * medium_theta_sd/ np.sqrt(medium_n),  medium_theta_hat + 1.96 * medium_theta_sd/ np.sqrt(medium_n)]

[-7.101915143280456, -1.9104435221858527]

In [24]:
low_theta_hat = low_samples['elas'].mean() 
low_theta_sd = low_samples['elas'].std() 
low_n = len(low_samples['elas'])
[low_theta_hat - 1.96 * low_theta_sd/ np.sqrt(low_n),  low_theta_hat + 1.96 * low_theta_sd/ np.sqrt(low_n)]

[-3.1265090923609344, -1.2915373820958524]

In [25]:
high_theta_hat = high_samples['elas'].mean() 
high_theta_sd = high_samples['elas'].std() 
high_n = len(high_samples['elas'])
[high_theta_hat - 1.96 * high_theta_sd/ np.sqrt(high_n),  high_theta_hat + 1.96 * high_theta_sd/ np.sqrt(high_n)]

[-3.8782429101459757, -1.8180680868633015]

In [26]:
[medium_theta_hat, low_theta_hat, high_theta_hat ]

[-4.5061793, -2.2090232, -2.8481555]

In [29]:
product_data = pd.read_csv(pyblp.data.BLP_PRODUCTS_LOCATION)
product_data.head()
product_formulations = (
   pyblp.Formulation('0 + prices + hpwt + air + mpd + space'), #1 + hpwt + air + mpd + space
   pyblp.Formulation('1 + prices + hpwt + air + mpd + space '), #
   #pyblp.Formulation('1 + log(hpwt) + air + log(mpg) + log(space) + trend')
)
product_formulations


(prices + hpwt + air + mpd + space, 1 + prices + hpwt + air + mpd + space)

In [30]:
mc_integration = pyblp.Integration('monte_carlo', size=50, specification_options={'seed': 0})
bfgs = pyblp.Optimization('bfgs', {'gtol': 1e-4})
mc_problem = pyblp.Problem(product_formulations, product_data, integration=mc_integration)

Initializing the problem ...
Initialized the problem after 00:00:00.

Dimensions:
 T    N     F    I     K1    K2    MD 
---  ----  ---  ----  ----  ----  ----
20   2217  26   1000   5     6     12 

Formulations:
       Column Indices:           0       1      2     3     4      5  
-----------------------------  ------  ------  ----  ---  -----  -----
 X1: Linear Characteristics    prices   hpwt   air   mpd  space       
X2: Nonlinear Characteristics    1     prices  hpwt  air   mpd   space


In [31]:
initial_sigma = np.diag([3.612, -0.1, 4.628, 1.818, 1.050, 2.056]) #

In [ ]:
results = mc_problem.solve(
    initial_sigma,
    optimization=bfgs
)

In [33]:
blp_elas = results.compute_elasticities()

Computing elasticities with respect to prices ...
Finished after 00:00:00.



In [37]:
blp_data = pd.read_csv(pyblp.data.BLP_PRODUCTS_LOCATION)
Js = blp_data.groupby(['market_ids']).size().tolist() 
product_id_list = []
for m in range(len(Js)):
    product_id_list.extend(range(Js[m]))  

blp_data['id_in_market'] = product_id_list
blp_elas_df = pd.DataFrame(blp_elas, columns=range(blp_elas.shape[1]))

# Add a column for the row indices
blp_elas_df['row_index'] = range(blp_elas.shape[0])

# Reshape the dataframe so that the column indices are in a column called 'column_index'
blp_elas_df = blp_elas_df.melt(id_vars='row_index', var_name='column_index', value_name='value')

blp_elas_df.columns = ['i','j','blp_elasticity']
blp_elas_df = blp_elas_df.loc[blp_elas_df['blp_elasticity'].isna() == False]

blp_elas_df = pd.merge(blp_elas_df, blp_data[['id_in_market', 'market_ids']],
                       how ='left', left_on =['i'], right_on = blp_data.index)


In [53]:
blp_self_elas_df = blp_elas_df.loc[blp_elas_df.j == blp_elas_df.id_in_market].copy()

In [54]:
blp_self_elas_df = blp_self_elas_df.sort_values(by=['i']).reset_index(drop = True)

In [56]:
df['blp_elas'] = blp_self_elas_df['blp_elasticity']

In [85]:
df_high = df.loc[df.type =='high'].copy()
df_medium = df.loc[df.type =='medium'].copy()
df_low = df.loc[df.type =='low'].copy()

np.random.seed(42)

high_samples = df_high.groupby('market_id').apply(lambda x: x.sample(1))
medium_samples = df_medium.groupby('market_id').apply(lambda x: x.sample(1))
low_samples = df_low.groupby('market_id').apply(lambda x: x.sample(1))

In [86]:
medium_theta_hat = medium_samples['elas'].mean() 
medium_theta_sd = medium_samples['elas'].std() 
medium_blp = medium_samples['blp_elas'].mean() 
medium_n = len(medium_samples['elas'])
print('CI', [medium_theta_hat - 1.96 * medium_theta_sd/ np.sqrt(medium_n),  medium_theta_hat + 1.96 * medium_theta_sd/ np.sqrt(medium_n)])
print('Our', medium_theta_hat, 'BLP', medium_blp)

CI [-4.92608334766917, -2.5169222141594427]
Our -3.7215028 BLP -3.7354352343785613


In [87]:
low_theta_hat = low_samples['elas'].mean() 
low_theta_sd = low_samples['elas'].std() 
low_n = len(low_samples['elas'])
low_blp = low_samples['blp_elas'].mean() 
print('CI', [low_theta_hat - 1.96 * low_theta_sd/ np.sqrt(low_n),  low_theta_hat + 1.96 * low_theta_sd/ np.sqrt(low_n)])
print('Our', low_theta_hat, 'BLP', low_blp)

CI [-3.1980469895183763, -0.9415156172931471]
Our -2.0697813 BLP -2.917415441728843


In [88]:
high_theta_hat = high_samples['elas'].mean() 
high_theta_sd = high_samples['elas'].std() 
high_n = len(high_samples['elas'])
high_blp = high_samples['blp_elas'].mean() 

print('CI',[high_theta_hat - 1.96 * high_theta_sd/ np.sqrt(high_n),  high_theta_hat + 1.96 * high_theta_sd/ np.sqrt(high_n)])
print('Our', high_theta_hat, 'BLP', high_blp)

CI [-6.220353651341485, -2.164058160487128]
Our -4.192206 BLP -5.670528130705975


In [89]:
[df_high['blp_elas'].mean() , df_medium['blp_elas'].mean() , df_low['blp_elas'].mean() ]

[-5.3100426714831395, -3.8885353403379748, -2.7865358482448497]